In [1]:
import pandas as pd 
import numpy as np 

In [2]:
df = pd.read_csv('../../../data/interim/up_cleaned_groundwater.csv')

In [3]:
df.shape

(2743299, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2743299 entries, 0 to 2743298
Data columns (total 8 columns):
 #   Column                                        Dtype  
---  ------                                        -----  
 0   Station                                       object 
 1   State                                         object 
 2   District LGD Code                             int64  
 3   District                                      object 
 4   Latitude                                      float64
 5   Longitude                                     float64
 6   Data Acquisition Time                         object 
 7   Groundwater Level Telemetry 6 Hourly (meter)  float64
dtypes: float64(3), int64(1), object(4)
memory usage: 167.4+ MB


In [5]:
# Date-Time Conversion:

df["Data Acquisition Time"] = pd.to_datetime(df["Data Acquisition Time"]) 
df.dtypes

Station                                                 object
State                                                   object
District LGD Code                                        int64
District                                                object
Latitude                                               float64
Longitude                                              float64
Data Acquisition Time                           datetime64[ns]
Groundwater Level Telemetry 6 Hourly (meter)           float64
dtype: object

In [6]:
#F-1: Calendar-Based Features: 

df["Year"] = df["Data Acquisition Time"].dt.year
df["Month"] = df["Data Acquisition Time"].dt.month
df["Day"] = df["Data Acquisition Time"].dt.day
df["Day_of_Week"] = df["Data Acquisition Time"].dt.dayofweek
df["Week_of_Year"] = df["Data Acquisition Time"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Data Acquisition Time"].dt.quarter
df['Is_Weekend'] = (df['Day_of_Week'] >= 5).astype(int) 

In [7]:
df[
    [
        "Data Acquisition Time",
        "Year",
        "Month",
        "Day",
        "Day_of_Week",
        "Week_of_Year",
        "Quarter",
        "Is_Weekend",
    ]
].head()

,Data Acquisition Time,Year,Month,Day,Day_of_Week,Week_of_Year,Quarter,Is_Weekend
0,2024-02-23 18:00:00,2024,2,23,4,8,1,0
1,2024-02-24 00:00:00,2024,2,24,5,8,1,1
2,2024-02-24 06:00:00,2024,2,24,5,8,1,1
3,2024-02-24 12:00:00,2024,2,24,5,8,1,1
4,2024-02-24 18:00:00,2024,2,24,5,8,1,1


In [8]:
#F-2: Season Feature: 

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

df["Season"] = df["Month"].apply(get_season)
df['Season']

0                Winter
1                Winter
2                Winter
3                Winter
4                Winter
               ...     
2743294    Post-Monsoon
2743295    Post-Monsoon
2743296    Post-Monsoon
2743297    Post-Monsoon
2743298    Post-Monsoon
Name: Season, Length: 2743299, dtype: object

In [9]:
# now since the data contains measurements from multiple stations. If we create lag features directly, the previous row for one station could come from a completely different station, which would produce incorrect values 
#so we will sort the data on the basis of 'Station' and 'Data Acquisition Time' 

df = df.sort_values(
    by=['Station', 'Data Acquisition Time']
).reset_index(drop=True)

In [10]:
df[['Station', 'Data Acquisition Time']].head(20)

,Station,Data Acquisition Time
0,390713- Jairampur PMS -Shallow,2024-02-23 18:00:00
1,390713- Jairampur PMS -Shallow,2024-02-24 00:00:00
2,390713- Jairampur PMS -Shallow,2024-02-24 06:00:00
3,390713- Jairampur PMS -Shallow,2024-02-24 12:00:00
4,390713- Jairampur PMS -Shallow,2024-02-24 18:00:00
5,390713- Jairampur PMS -Shallow,2024-02-25 00:00:00
6,390713- Jairampur PMS -Shallow,2024-02-25 06:00:00
7,390713- Jairampur PMS -Shallow,2024-02-25 12:00:00
8,390713- Jairampur PMS -Shallow,2024-02-25 18:00:00
9,390713- Jairampur PMS -Shallow,2024-02-26 00:00:00


In [11]:
#F-3: Lag Features: 
# Groundwater levels are highly autocorrelated, so previous values are often the strongest predictors of future values

In [12]:
#Lag-1: 

df['GW_Lag_1'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .shift(1)
)
df['GW_Lag_1']

0              NaN
1           -0.247
2           -0.256
3           -0.252
4           -0.167
            ...   
2743294   -108.126
2743295   -108.091
2743296   -108.052
2743297   -107.898
2743298   -108.116
Name: GW_Lag_1, Length: 2743299, dtype: float64

In [13]:
#Lag-2: 

df['GW_Lag_2'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .shift(2)
) 
df['GW_Lag_2']

0              NaN
1              NaN
2           -0.247
3           -0.256
4           -0.252
            ...   
2743294   -108.061
2743295   -108.126
2743296   -108.091
2743297   -108.052
2743298   -107.898
Name: GW_Lag_2, Length: 2743299, dtype: float64

In [14]:
#Lag-3: 

df['GW_Lag_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .shift(3)
) 
df['GW_Lag_3']

0              NaN
1              NaN
2              NaN
3           -0.247
4           -0.256
            ...   
2743294   -108.044
2743295   -108.061
2743296   -108.126
2743297   -108.091
2743298   -108.052
Name: GW_Lag_3, Length: 2743299, dtype: float64

In [15]:
#F-4: Rolling Statistics Features: 

In [16]:
# Rolling Mean (Window = 3)

df['GW_Rolling_Mean_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .transform(lambda x: x.rolling(window=3).mean())
)
df['GW_Rolling_Mean_3']

0                 NaN
1                 NaN
2           -0.251667
3           -0.225000
4           -0.225000
              ...    
2743294   -108.092667
2743295   -108.089667
2743296   -108.013667
2743297   -108.022000
2743298   -108.006333
Name: GW_Rolling_Mean_3, Length: 2743299, dtype: float64

In [17]:
# Rolling Standard Deviation (Window = 3) 

df['GW_Rolling_STD_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .transform(lambda x: x.rolling(window=3).std())
)
df['GW_Rolling_STD_3']

0               NaN
1               NaN
2          0.004509
3          0.050269
4          0.050269
             ...   
2743294    0.032532
2743295    0.037018
2743296    0.102051
2743297    0.112054
2743298    0.109006
Name: GW_Rolling_STD_3, Length: 2743299, dtype: float64

In [18]:
df['GW_Expanding_Mean'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .transform(lambda x: x.expanding().mean())
)
df['GW_Expanding_Mean']

0         -0.247000
1         -0.251500
2         -0.251667
3         -0.230500
4         -0.235600
             ...   
2743294   -8.310654
2743295   -8.398609
2743296   -8.486274
2743297   -8.573976
2743298   -8.661427
Name: GW_Expanding_Mean, Length: 2743299, dtype: float64

This feature answers:

"What has been the average groundwater level at this station up to the current observation?"

Unlike a rolling mean, which only looks at the last few observations, the expanding mean summarizes the station's historical behavior.

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2743299 entries, 0 to 2743298
Data columns (total 22 columns):
 #   Column                                        Dtype         
---  ------                                        -----         
 0   Station                                       object        
 1   State                                         object        
 2   District LGD Code                             int64         
 3   District                                      object        
 4   Latitude                                      float64       
 5   Longitude                                     float64       
 6   Data Acquisition Time                         datetime64[ns]
 7   Groundwater Level Telemetry 6 Hourly (meter)  float64       
 8   Year                                          int32         
 9   Month                                         int32         
 10  Day                                           int32         
 11  Day_of_Week             

In [20]:
df.to_csv('../../../data/processed/up_featured_groundwater.csv')